FRAME TO FRAME INFERENCE & EVALUATION

In [33]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
from spicy import signal
import pickle
import warnings
import gzip
import scipy.io
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

# 8k Net trained
workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
# 16k Net trained
# workspace_dir = '/home/adelval/BTS/TFM/test/'

reduced_net = False

sys.path.append(workspace_dir + 'src/net')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [34]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [35]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length without extending to slide last frame
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010, Mw=20):
    # Mw limit the number of windows
    N = int(Ns * fs)            # Number of samples in each window 640
    M = int(Ms * fs)            # Step size (number of samples between window starts) 160
    n = (len(x) + M - 1) // M   # Number of frames 23
    # print("Number of frames", n)    
    # T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    # Se ignora el padding porque la ventana se deslizará
    # if T > len(x):
    #     print("rellena con ceros")
    #     xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, Mw*M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    xa = xa[ind.astype(int).T].astype(np.float32)
    # print(f'Window frames {xa[19,:5]}')

    return xa

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[1])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=1)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:, :(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


def frame_fft(data, fs, w, nfft, max_windows):
    
    N = int(w * fs)
    F =[int(2 ** np.ceil(np.log2( N)))//2]

    x = offset(data)

    # Emphasis to increase the amplitude of high freq
    x = preemphasis(x)

    X = hamming(windowing2(x, fs=fs, Ns=w, Ms=m, Mw=max_windows))
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])
    # print(f"la shape de Xfft es {Xfft.shape}")
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    
    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    # print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [36]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


def frame_fb_mfcc(data, fs, B, w, nfft, max_windows):

    N = int(w * fs)
    F = [int(nffti/2) for nffti in nfft]
    fb_time = time.time()
    fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
    # print(f'El tiempo de cálculo de los filtros es {time.time() - fb_time}')
    # dct_time = time.time()
    dct = [ f_base_dct(Bi) for Bi in B] 
    # print(f'El tiempo de cálculo de las bases dct es {time.time() - dct_time}')

    x = offset(data)
    x = preemphasis(x)
    
    X = hamming(windowing2(x, fs=fs, Ns=w, Ms=m, Mw=max_windows))
    
    Xfft = fft(X, nfft[0])
    # print(f'la shape en FCMFCC de Xfft es {Xfft.shape}')
    # fb_time = time.time()
    Xb = np.log(Xfft.dot( fb[0] ) + 1)
    # print(f'El tiempo de cálculo de los coeficientes fb es {time.time() - fb_time}')
    # dct_time = time.time()
    Xc = Xb.dot(dct[0])   
    # print(f'El tiempo de cálculo de los coeficientes dct es {time.time() - dct_time}')                             
    
    X = np.concatenate( [Xb, Xc], 1 )
    
    X = np.asarray(X, dtype=np.float32)
    
    # print("El tamaño de x08k2 es: ",XX.shape)
    # print("Vector con FilterBank MFCC del frame: \n", X[:10])
    
    return X

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc):
    # Normalization of fbmfcc

    file = workspace_dir + 'data/model/fe1_norm1.pkl'  
    # file = workspace_dir + 'data/model/fe1_norm1_rn.pkl'  
    x = frame_fbmfcc
    mu, std = read_pkl(file)
    x -= mu
    x /= std + 1e-6
    # print("Vector con FB MFCC normalizado del frame: \n", x[:10])

    return x
    

In [37]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

if (reduced_net):
    input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions_rn.pkl') 
else:
    input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 


print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append( workspace_dir + 'src/net')

if (reduced_net):
    from net_snr import Net_snr
else:
    from net_snr_original import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=True, single_gpu=True)

if (reduced_net):
    net_snr.load_theta( workspace_dir + 'data/model/theta_last_rn')
else:
    net_snr.load_theta( workspace_dir + 'data/model/theta_last')


  input_dim: 576
  output_dim: 512

  Net_snr:
    nb_params: 29.99M
    cuda: True
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/afterburner8k/data/model/theta_last


In [38]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    n_frames, fft_fb = x.shape
    x = x.reshape(1, n_frames, fft_fb)

    snr = net_snr.predict(x)
    snr = to_numpy(snr.squeeze())

    return snr


In [39]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        # print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        # print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        # print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        # print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        # print(f'El tamaño de la salida del filtro sera {outf.shape}')
        # print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        # print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        # print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    snr_net = snr_net.reshape(-1,1) # para darle 2-D
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    # print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


In [40]:
import cProfile, pstats, io
sys.path.append('/home/adelval/BTS/TFM/quality')
from srmr import *


# Parámetros
fs=8000
B=[32]
w=[0.030, 0.040, 0.050]
m=0.01
nfft=[1024]
gmin = 0.0562
min_windows = 4
max_windows = 20
diezmation_factor = [1,2,4,8]


## SELECCIÓN DE AUDIO
# AUDIOS 16K
# x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/7-CH0_C01_city_5dB.wav']
# x_test = ['/home/adelval/BTS/TFM/audios/livekit/audio_received.wav']
# x_test = ['/home/adelval/BTS/TFM/test/data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner_2023/afterburner16k/data/audio/callcenter_movistar/AUDIOS/Audio-AudioModule_708351_AudioChannel_22403522_19-May-2023_15.53.49.097.wav']

# AUDIOS 8K
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav']
x_test = ['/home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/10-CH0_C01_airport_10dB.wav']
# x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/11-CH0_C01_construction_5dB.wav']
# x_test = [  '/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav',
#             '/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/6-CH0_C01_traffic_15dB.wav',
#             '/home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/10-CH0_C01_airport_10dB.wav',
#             '/home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/12-CH0_C01_babies_5dB.wav',
#             '/home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/14-CH0_C01_callcenters_5dB.wav', 
#             '/home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/16-CH0_C01_barks_5dB.wav']

# x_test = ['/home/adelval/BTS/TFM/audios/Movistar_PrePoc_1ch/36635c2b-ca8a-5d7c-abb6-42981f7613e9.wav']

srmr_value = np.zeros((len(diezmation_factor)*len(w)+1,len(x_test)))


for audio_cnt in range(len(x_test)):
    metrics_it = 0

    print(f'\n+{"-" * 46}AUDIO {audio_cnt}{"-" * 46}+\n {x_test[audio_cnt]}')
    print(f'+{"-" * 99}+\n')

    audio, fs = read_audio(x_test[audio_cnt])
    # print(f'La frecuencia de muestreo es {fs} Hz')
    print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

    try:
        srmr_value_before, _ = srmr(audio, fs, n_cochlear_filters=23, low_freq=125, min_cf=4, max_cf=128, fast=True, norm=False)
        srmr_value[metrics_it, audio_cnt] = np.round(srmr_value_before,2)
    except Exception as e:
        srmr_value[metrics_it, audio_cnt] = np.nan
        print(f'Error in SRMR calculation: {e}')
    print(f'SRMR: {srmr_value[metrics_it, audio_cnt]} before enhancement')



    for w_i in w:

        for diezmation_factor_i in diezmation_factor:
            if(reduced_net):
                output_enh_file = os.path.basename(x_test[audio_cnt])
                output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_RN_HM_'+ str(diezmation_factor_i) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                            +'w_'+ str(int(w_i*1000)) +'ms_s'+ str(int(m*1000)) +'_int16.wav')
                print(f'El audio mejorado {output_enh_int}')
            else:
                output_enh_file = os.path.basename(x_test[audio_cnt])
                output_enh_int = "/home/adelval/BTS/TFM/audios/enh/"+output_enh_file.replace('.wav','_f2f_enh_HM_'+ str(diezmation_factor_i) + '_' + str(int(fs/1000)) +'k_'+ str(min_windows) +'to'+ str(max_windows) 
                            +'w_'+ str(int(w_i*1000)) +'ms_s'+ str(int(m*1000)) +'_int16.wav')
                print(f'\nEl audio mejorado es {output_enh_int}')



            #------------------------SNR PRE-----------------------------#
            # pre_vad = compute_vad(audio, fs, w_i, m, nfft[0])
            # snr_prev = int(wada_snr(audio, fs, pre_vad))
            # print('snr(wada)=%idB, file: %s' % (snr_prev, x_test[audio_cnt]))
            #------------------------------------------------------------#
            metrics_it += 1
            diff_acum = 0
            diff_inf_acum = 0

            
            frame_size = 0.01  # 10 ms
            frame_samples = int(frame_size * fs)  # Muestras por frame
            
            shift_size = m  # 10 ms
            shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
            
            window_size = w_i  # 40 ms (640 muestras)
            window_samples = int(window_size * fs)  # Muestras por ventana 640
            
            window_inference_min = int((min_windows + (window_size/frame_size)-1) * frame_samples)
            window_inference_max = int((max_windows + (window_size/frame_size)-1) * frame_samples)
            buffer_frame = np.zeros(0)  # Buffer de ventana recibida

            print(f'+{"-" * 52}+')
            print(f'  PARÁMETROS SELECCIONADOS')
            print(f'+{"-" * 52}+')
            print(f'| Duración de frame:        {frame_size*1000:6.1f} ms{" " * 16}|')
            print(f'| Desplazamiento de:        {shift_size*1000:6.1f} ms{" " * 16}|')
            print(f'| Duración de la ventana:   {window_size*1000:6.1f} ms{" " * 16}|')
            print(f'| El buffer retendra desde   {min_windows:2}-{max_windows:2} ventanas{" " * 10}|')
            print(f'| Factor de diezmado:        {diezmation_factor_i:2}{" " * 22}|')
            print(f'+{"-" * 52}+')

            it = 0
            snr_frame_mask = np.ones((512,min_windows)) # Inicializado con la duración de la ventana de inferencia
            yenh = np.zeros(len(audio)) # Inicializado con la duración del audio original

            # CALCULO DE LA MÁSCARA SNR 
            for n_frame in range(int((len(audio)/fs)*100)):
                # Obtener el frame de audio
                frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]
                # print(f'{n_frame} frame de {len(frame)} --> {frame[:10]}')
                # Concatenar el frame recibido al buffer `buffer_frame`
                buffer_frame = np.concatenate([buffer_frame, frame])
                start_time = time.time()

                if len(buffer_frame) >= window_samples:
                    # Extraer las primeras 640 muestras como ventana completa
                    if len(buffer_frame) < window_inference_max:
                        work_window = buffer_frame[it*shift_samples:it*shift_samples+window_samples]
                        it += 1 # Number of windows received
                        if len(buffer_frame) >= window_inference_min:
                            # print("Reached MIN WINDOW --> STRATING INFERENCE")
                            work_inf_frames = buffer_frame[:window_inference_max]
                            # start_fft = time.time()
                            fft_windows = frame_fft(work_inf_frames, fs, w_i, nfft, it)
                            # end_fft = time.time()
                            # print(f'Tiempo de procesamiento de la FFT del frame {n_frame} es de {end_fft-start_fft} segundos')

                            # start_log = time.time()
                            fft_windows_log = log_scale(fft_windows)
                            # end_log = time.time()
                            # print(f'Tiempo de procesamiento de la escala log del frame {n_frame} es de {end_log-start_log} segundos')

                            # start_fb = time.time()
                            fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w_i, nfft, it)
                            # end_fb = time.time()
                            # print(f'Tiempo de procesamiento de la FB MFCC del frame {n_frame} es de {end_fb-start_fb} segundos')

                            # start_norm = time.time()
                            fb_windows_norm = norm_fb_frame(fb_windows)
                            # end_norm = time.time()
                            # print(f'Tiempo de procesamiento de la normalización del frame {n_frame} es de {end_norm-start_norm} segundos')

                            windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )
                            # start_prof = time.time()
                            # Process inference with factor 2 diezmation
                            if(n_frame % diezmation_factor_i == 0):
                                snr_frame_mask = net_eval(windows_concat)
                                snr_frame_mask = snr_frame_mask.T
                            # end_prof = time.time()
                            # print(f'Tiempo de procesamiento de la inferencia del frame {n_frame} es de {end_prof-start_prof} segundos')

                    # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
                    else:
                        work_window = buffer_frame[window_inference_max-window_samples:] # Los últimos frames del buffer
                        work_inf_frames = buffer_frame[:window_inference_max]
                        fft_windows = frame_fft(work_inf_frames, fs, w_i, nfft, max_windows)
                        fft_windows_log = log_scale(fft_windows)
                        fb_windows = frame_fb_mfcc(work_inf_frames, fs, B, w_i, nfft, max_windows)
                        fb_windows_norm = norm_fb_frame(fb_windows)
                        windows_concat = np.concatenate( (fft_windows_log,fb_windows_norm), 1 )

                        # start_prof = time.time()
                        # Process inference with factor 2 diezmation
                        if(n_frame % diezmation_factor_i == 0):
                            snr_frame_mask = net_eval(windows_concat)
                            snr_frame_mask = snr_frame_mask.T
                        # end_prof = time.time()
                        # print(f'Tiempo de procesamiento de la inferencia del frame {n_frame} es de {end_prof-start_prof} segundos')
                        #Desplazar las muestras en `buffer_frame` para la próxima ventana
                        buffer_frame = buffer_frame[shift_samples:]

                    # Evaluacion con la máscara pertinente (para las primeras 3 ventanas sin máscara calculada)
                    # cnt = int(n_frame - w[0]/m) Ajustar al tamaño de la ventana
                    cnt = int(n_frame - (w_i/m-1))
                    # print(f'EVALUATION OF WINDOW {cnt}')
                    x = np.array(work_window, dtype=np.float32) / 2 ** 15 # 0.04 * fs = 640 samples
                    xenh, filt = noiseReduction(x, snr_frame_mask[:,-1], fs, window_samples, shift_samples, nfft[0], gmin)
                    # print(f'Las dimensiones del filtro son {filt.shape}')
                    slice_size = min(len(yenh) - cnt * shift_samples, window_samples)
                    yenh[cnt * shift_samples : cnt * shift_samples + slice_size] += xenh[0:slice_size]

                # end_time = time.time()
                # diff_acum += end_time - start_time
                # print(f'Tiempo de procesamiento del frame {n_frame} es de {end_time-start_time} segundos')
            yenh_clipped = np.clip(yenh*2**15, -32768, 32767).astype(np.int16)

            try:
                srmr_value_after, _ = srmr(yenh_clipped, fs, n_cochlear_filters=23, low_freq=125, min_cf=4, max_cf=128, fast=True, norm=False)
                srmr_value[metrics_it, audio_cnt] = np.round(srmr_value_after,2)
            except Exception as e:
                srmr_value[metrics_it, audio_cnt] = np.nan
                print(f'Error in SRMR calculation: {e}')
            
            
            print(f'SRMR: {srmr_value[metrics_it, audio_cnt]} after enhancement\n')

            wavfile.write(output_enh_int,fs,yenh)

    # print(f'El tiempo medio de procesamiento es de {diff_acum/n_frame} segundos')





    #--------------------------SNR POST--------------------------#
    # post_vad = compute_vad(yenh_clipped, fs, w[0], m, nfft[0])
    # snr_post = int(wada_snr(yenh_clipped, fs, pre_vad))
    # print('snr(wada)=%idB, file: %s' % (snr_post, output_enh))
    #------------------------------------------------------------#


print(srmr_value)


+----------------------------------------------AUDIO 0----------------------------------------------+
 /home/adelval/BTS/TFM/afterburner8k_rn_2/data/audio/minitest_8k/10-CH0_C01_airport_10dB.wav
+---------------------------------------------------------------------------------------------------+

La duración del audio es 3.070125 segundos y 24561 muestras
SRMR: 6.9 before enhancement

El audio mejorado es /home/adelval/BTS/TFM/audios/enh/10-CH0_C01_airport_10dB_f2f_enh_HM_1_8k_4to20w_30ms_s10_int16.wav
+----------------------------------------------------+
  PARÁMETROS SELECCIONADOS
+----------------------------------------------------+
| Duración de frame:          10.0 ms                |
| Desplazamiento de:          10.0 ms                |
| Duración de la ventana:     30.0 ms                |
| El buffer retendra desde    4-20 ventanas          |
| Factor de diezmado:         1                      |
+----------------------------------------------------+
SRMR: 9.17 after enhance

In [ ]:
# Number of parameters and audio files
num_params, num_audios = srmr_value.shape

# Create the x-axis labels for audios
# x_labels = [f"Audio {i+1}" for i in range(num_audios)]
x_labels = ['5-CH0_C01_stadium_15dB.wav',
            '6-CH0_C01_traffic_15dB.wav',
            '10-CH0_C01_airport_10dB.wav',
            '12-CH0_C01_babies_5dB.wav',
            '14-CH0_C01_callcenters_5dB.wav',
            '16-CH0_C01_barks_5dB.wav']

parameters = ["Original Audio", "30_1", "30_2", "30_4", "30_8", 
              "40_1", "40_2", "40_4", "40_8", "50_1", "50_2", "50_4", "50_8"]  # Replace with your parameters
assert len(parameters) == srmr_value.shape[0], "Number of parameters must match the number of rows in srmr_value"

# Use a colormap to generate distinct colors
cmap = plt.cm.get_cmap('tab20', num_params)  # Use 'tab20' for up to 20 distinct colors


# Plot each parameter's SRMR values
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed

for i, param_name in enumerate(parameters):
    plt.plot(range(num_audios), srmr_value[i, :], marker='o', label=param_name, color=cmap(i))

# Add labels, title, and legend
plt.xticks(range(num_audios), x_labels, rotation=45)  # Rotate x-axis labels for clarity
plt.xlabel("Audios")
plt.ylabel("SRMR")
plt.title("SRMR Values for Different Audios and Parameters")
plt.legend(loc="upper left", bbox_to_anchor=(1.05, 1))  # Place the legend appropriately
plt.grid(True)  # Add grid for better readability

# Show the plot
plt.tight_layout()
plt.show()